In [0]:
# Predictions are made BEFORE the true next-session close exists. Once it exists in the feature table
# (as target_next_close of the row the prediction was made from), copy it into the inference log so the
# regression monitor can compute MAE / RMSE / R2 per model version.
import sys
import os
sys.path.insert(0, os.getcwd())

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from utils import load_project, table

cfg = load_project()
dbutils.widgets.text("catalog", "stock_demo")
catalog = dbutils.widgets.get("catalog")

log_table = table(cfg, "inference_log", catalog)
if not spark.catalog.tableExists(log_table):
    print(f"{log_table} does not exist yet (run batch_inference first) - nothing to back-fill.")
    dbutils.jobs.taskValues.set(key="rows_backfilled", value=0)
    dbutils.notebook.exit("no inference log")

labels = (spark.table(table(cfg, "features", catalog))
          .select("symbol", "trade_ts", F.col("target_next_close").alias("_label"))
          .filter(F.col("_label").isNotNull()))

before = spark.table(log_table).filter(F.col("actual_next_close").isNull()).count()
(DeltaTable.forName(spark, log_table).alias("t")
    .merge(labels.alias("s"), "t.symbol = s.symbol AND t.trade_ts = s.trade_ts AND t.actual_next_close IS NULL")
    .whenMatchedUpdate(set={"actual_next_close": "s._label"})
    .execute())
after = spark.table(log_table).filter(F.col("actual_next_close").isNull()).count()

print(f"Unlabelled rows before: {before} | after: {after} | back-filled: {before - after}")
dbutils.jobs.taskValues.set(key="rows_backfilled", value=int(before - after))